In [22]:
import os
print("bench.py exists:", os.path.exists("bench.py"))
print("prompts.txt exists:", os.path.exists("prompts.txt"))

bench.py exists: True
prompts.txt exists: True


In [12]:

!find / -name "bench.py" 2>/dev/null
!find / -name "prompts.txt" 2>/dev/null

/usr/local/lib/python3.13/dist-packages/bottleneck/benchmark/bench.py
/tools/google-cloud-sdk/lib/third_party/chardet/bench.py
/tools/google-cloud-sdk/platform/gsutil/third_party/chardet/bench.py


In [19]:

!git clone https://github.com/code2expert/ai-datacenter-bootcamp-labs.git

!find ai-datacenter-bootcamp-labs -iname "bench*.py"

!find ai-datacenter-bootcamp-labs -maxdepth 3 -type d

Cloning into 'ai-datacenter-bootcamp-labs'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 97 (delta 21), reused 84 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 103.83 KiB | 878.00 KiB/s, done.
Resolving deltas: 100% (21/21), done.
ai-datacenter-bootcamp-labs/w3d5-benchmark-harness/bench.py
ai-datacenter-bootcamp-labs
ai-datacenter-bootcamp-labs/w3d4-quantise-and-lock
ai-datacenter-bootcamp-labs/shared
ai-datacenter-bootcamp-labs/w3d2-inference-anatomy
ai-datacenter-bootcamp-labs/w3d3-engine-swap
ai-datacenter-bootcamp-labs/w2d5-compose-and-teams
ai-datacenter-bootcamp-labs/w2d3-containerise
ai-datacenter-bootcamp-labs/.git
ai-datacenter-bootcamp-labs/.git/hooks
ai-datacenter-bootcamp-labs/.git/info
ai-datacenter-bootcamp-labs/.git/branches
ai-datacenter-bootcamp-labs/.git/objects
ai-datacenter-bootcamp-labs/.git/objects/pack
ai-datacenter-bootcamp-labs/.git/obj

In [23]:
import shutil
shutil.copy("ai-datacenter-bootcamp-labs/w3d5-benchmark-harness/bench.py", "bench.py")

import os
print("bench.py exists now:", os.path.exists("bench.py"))

bench.py exists now: True


In [24]:
from google.colab import files
uploaded = files.upload()

Saving prompts.txt to prompts (1).txt


In [25]:
!sudo apt-get update -y
!sudo apt-get install -y python3.10 python3.10-venv python3.10-dev
!python3.10 -m venv /content/venv
!/content/venv/bin/python -m pip install --upgrade pip

VENV_PYTHON = "/content/venv/bin/python"

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,184 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Fetched 14.0 MB in 2s (7,032 kB/s)
Reading package lists... Done

In [31]:
!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=81.56 ttft_p95=0.078 errors=0
[level 2] tok/s=160.09 ttft_p95=0.1161 errors=0
[level 4] tok/s=275.52 ttft_p95=0.146 errors=0
[level 8] tok/s=462.94 ttft_p95=0.1718 errors=0
[level 16] tok/s=654.03 ttft_p95=0.2906 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     81.56      0.054      0.078     1.590    20     0
   2    160.09      0.053      0.116     1.601    20     0
   4    275.52      0.066      0.146     1.742    20     0
   8    462.94      0.113      0.172     1.989    20     0
  16    654.03      0.283      0.291     2.578    20     0

wrote bench_report.json (run appended)


In [26]:
import subprocess, sys

VLLM_PIN = "0.6.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
AUTOAWQ_PIN = "0.2.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [VENV_PYTHON, "-m", "pip", "install", "-q", *specs]
    print("installing (venv):", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)
print("serving pins installed inside the venv (with AWQ)")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"httpx=={HTTPX_PIN}", f"openai=={OPENAI_PIN}"], check=True)
print("client-side pins installed in the notebook kernel")



installing (venv): vllm==0.6.* transformers==4.46.* accelerate==1.1.* autoawq==0.2.* httpx==0.27.* openai==1.54.*
serving pins installed inside the venv (with AWQ)
client-side pins installed in the notebook kernel


In [27]:
!/content/venv/bin/python -c "import torch; print('cuda available:', torch.cuda.is_available())"


cuda available: True


In [28]:
import os, signal, subprocess

PORT = 8000
SERVER_LOG = "/content/server.log"
SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = [VENV_PYTHON, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        cmd += [k] if v is None else [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT,
                            start_new_session=True)
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()


launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 9756, logging to /content/server.log


In [35]:
import json
levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"ttft_p95={L['ttft_p95_s']:.3f}  lat_p95={L['latency_p95_s']:.3f}  "
          f"errors={L['errors']}")

TARGET_P95_S = 2.0
under = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda L: L["concurrency"]) if under else None
print("knee:", knee)

c= 1  tok/s=   81.6  ttft_p95=0.078  lat_p95=1.590  errors=0
c= 2  tok/s=  160.1  ttft_p95=0.116  lat_p95=1.601  errors=0
c= 4  tok/s=  275.5  ttft_p95=0.146  lat_p95=1.742  errors=0
c= 8  tok/s=  462.9  ttft_p95=0.172  lat_p95=1.989  errors=0
c=16  tok/s=  654.0  ttft_p95=0.291  lat_p95=2.578  errors=0
knee: {'concurrency': 8, 'tokens_per_s': 462.94, 'ttft_p50_s': 0.1129, 'ttft_p95_s': 0.1718, 'latency_p95_s': 1.9889, 'errors': 0, 'ok': 20, 'wall_s': 4.327}


In [36]:
with open("knee.json", "w") as f:
    json.dump({"target_p95_s": TARGET_P95_S,
               "knee_concurrency": knee["concurrency"] if knee else None}, f)

In [39]:
from google.colab import files
uploaded = files.upload()

Saving capacity-note.md to capacity-note.md


In [54]:
%%writefile capacity-note.md
# Capacity note (team, one page)

## The numbers

- Locked model: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Target p95 end-to-end latency (your SLO today): `2.0` seconds
- Knee concurrency (highest concurrency whose p95 is still under target): `8`
- Tokens per second at the knee: `462.9`
- Max sustainable request rate at the target p95: `4.0 req/s` (concurrency 8 ÷ p95
  latency 1.989s at that level)

## The limiting family

One sentence, using this morning's triage lens (compute vs memory vs overhead):
which family limits this stack at the knee, and the tell that points to it.

- Memory-bound: throughput scales sub-linearly past c=4 (275.5 → 462.9 tok/s from
  c=4 to c=8, not the ~2x a compute-bound stage would show), and p95 climbs
  steadily alongside it (1.742s → 1.989s → 2.578s) while ttft_p95 nearly doubles
  from c=8 to c=16 (0.172s → 0.291s) — the KV-cache/decode memory-bandwidth
  ceiling saturating under concurrent batches, not a compute wall.

## Why the knee, not the peak

One sentence in your own words on why you report the knee at the SLO rather than
the peak throughput.

- The peak (654 tok/s at c=16) only exists because p95 already blew past the 2.0s
  SLO to 2.578s, so it counts requests that were served too slowly to promise;
  the knee at c=8 is the highest concurrency where the stack still honors the
  SLO, making it the number the team can actually commit to.

Overwriting capacity-note.md


In [55]:
print(open("capacity-note.md").read())

# Capacity note (team, one page)

## The numbers

- Locked model: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Target p95 end-to-end latency (your SLO today): `2.0` seconds
- Knee concurrency (highest concurrency whose p95 is still under target): `8`
- Tokens per second at the knee: `462.9`
- Max sustainable request rate at the target p95: `4.0 req/s` (concurrency 8 ÷ p95
  latency 1.989s at that level)

## The limiting family

One sentence, using this morning's triage lens (compute vs memory vs overhead):
which family limits this stack at the knee, and the tell that points to it.

- Memory-bound: throughput scales sub-linearly past c=4 (275.5 → 462.9 tok/s from
  c=4 to c=8, not the ~2x a compute-bound stage would show), and p95 climbs
  steadily alongside it (1.742s → 1.989s → 2.578s) while ttft_p95 nearly doubles
  from c=8 to c=16 (0.172s → 0.291s) — the KV-cache/decode memory-bandwidth
  ceiling saturating under concurrent batches, not a compute wall.

## Why the knee, not the peak

One sen

In [56]:
import re
content = open("capacity-note.md").read()
print(content)
print("---")
print("FILL: occurrences found at:")
for m in re.finditer(r"FILL:", content):
    start = max(0, m.start()-40)
    end = min(len(content), m.end()+20)
    print(repr(content[start:end]))

# Capacity note (team, one page)

## The numbers

- Locked model: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Target p95 end-to-end latency (your SLO today): `2.0` seconds
- Knee concurrency (highest concurrency whose p95 is still under target): `8`
- Tokens per second at the knee: `462.9`
- Max sustainable request rate at the target p95: `4.0 req/s` (concurrency 8 ÷ p95
  latency 1.989s at that level)

## The limiting family

One sentence, using this morning's triage lens (compute vs memory vs overhead):
which family limits this stack at the knee, and the tell that points to it.

- Memory-bound: throughput scales sub-linearly past c=4 (275.5 → 462.9 tok/s from
  c=4 to c=8, not the ~2x a compute-bound stage would show), and p95 climbs
  steadily alongside it (1.742s → 1.989s → 2.578s) while ttft_p95 nearly doubles
  from c=8 to c=16 (0.172s → 0.291s) — the KV-cache/decode memory-bandwidth
  ceiling saturating under concurrent batches, not a compute wall.

## Why the knee, not the peak

One sen

In [57]:
content = open("capacity-note.md").read()

replacements = {
    "FILL: the model id you benchmarked": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "FILL: e.g. `2.0` seconds": f"`{TARGET_P95_S}` seconds",
    "FILL: e.g. `8`": f"`{knee['concurrency']}`",
    "FILL: e.g. `410`": f"`{knee['tokens_per_s']}`",
}
print(content)

# Capacity note (team, one page)

## The numbers

- Locked model: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Target p95 end-to-end latency (your SLO today): `2.0` seconds
- Knee concurrency (highest concurrency whose p95 is still under target): `8`
- Tokens per second at the knee: `462.9`
- Max sustainable request rate at the target p95: `4.0 req/s` (concurrency 8 ÷ p95
  latency 1.989s at that level)

## The limiting family

One sentence, using this morning's triage lens (compute vs memory vs overhead):
which family limits this stack at the knee, and the tell that points to it.

- Memory-bound: throughput scales sub-linearly past c=4 (275.5 → 462.9 tok/s from
  c=4 to c=8, not the ~2x a compute-bound stage would show), and p95 climbs
  steadily alongside it (1.742s → 1.989s → 2.578s) while ttft_p95 nearly doubles
  from c=8 to c=16 (0.172s → 0.291s) — the KV-cache/decode memory-bandwidth
  ceiling saturating under concurrent batches, not a compute wall.

## Why the knee, not the peak

One sen

In [50]:
print(open("knee.json").read())

{"target_p95_s": 2.0, "knee_concurrency": 8}


In [58]:

import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

server healthy after about 0s: http://localhost:8000/v1/models -> 200


In [59]:
# Green-check verifier for Lab W3D5 (benchmark harness).
# Paste this as the last cell of your day-5 notebook and run it. It reads
# bench_report.json (from the harness) and capacity-note.md, and checks the
# schema, that at least four concurrency levels ran, that errors are zero or
# explained, and that the capacity note is filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) bench report
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    # bench.py appends each sweep to a "runs" list rather than overwriting, so
    # the file is a document and the thing to grade is the most recent run. A
    # bare list is also accepted, for a report assembled by hand.
    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output ({'runs': [...]}) "
             "or a bare list of per-level objects")
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    # 2) the knee file from Cell 5
    if not os.path.exists("knee.json"):
        fail("knee.json not found; write it in Cell 5")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number; set TARGET_P95_S to your "
             "real SLO before computing the knee (the 'target left at zero' "
             "failure mode)")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty: no level stayed under your target. "
             "Either your SLO is stricter than this stack can serve (explain "
             "that in the note) or the target was never set from the card")

    # errors must be zero, OR explained in the capacity note
    # 3) capacity note filled in
    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors in the sweep and no explanation in "
             "capacity-note.md; zero errors, or explain them")

    # sanity: throughput should be present and positive somewhere
    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0
               for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, "
          f"total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS
